# Week 2 — Data Collection, Preparation & Cleaning Notebook

**Project:** Agricultural Production, Yield & Agribusiness Intelligence — India  
**Purpose:** Demonstrate a reproducible data collection, profiling, cleaning, validation and transformation workflow.

This notebook uses the Week 1 DES normal-estimate extract as the **real working dataset** for the Week 2 pipeline. Future state/district APY, climate and market files can be added to the same raw-data structure without changing the overall workflow.

> **Important:** The current notebook does not pretend to have collected live Agmarknet or climate records. Those are documented as planned integration sources; only the DES extract actually present in this repository is processed here.


## 1. Data Collection Strategy

The collection hierarchy is:

1. Official API / machine-readable endpoint
2. Official CSV/Excel/bulk download
3. Official published report/table
4. Controlled web extraction where permitted and necessary
5. Manual extraction for small authoritative tables

The raw file is preserved unchanged. Every future source should be accompanied by retrieval date, source URL, filters, units and revision/version information.

The DES APY interface supports state, district, crop, season and year filtering, making it appropriate for the next regional stage. citeturn0search0turn0search1


## 2. Project Data Layout

```text
data/
├── raw/          # untouched source files
├── processed/    # cleaned / analysis-ready data
└── reference/    # crop, state, district and cleaning mappings

docs/
├── data_dictionary.csv
└── source_register.csv

scripts/
└── generate_quality_report.py

outputs/
└── quality_reports/
```


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE = Path("..")
RAW_PATH = BASE / "data" / "raw" / "des_normal_estimates_major_crops.csv"
PROCESSED_PATH = BASE / "data" / "processed" / "des_cleaned_major_crops.csv"

df_raw = pd.read_csv(RAW_PATH)
df_raw.head()


## 3. Source and Dataset Profile

In [ ]:
print("Rows:", len(df_raw))
print("Columns:", df_raw.shape[1])
print("Columns:", list(df_raw.columns))
print("Crops:", df_raw["crop"].nunique())
print("Periods:", df_raw["period"].nunique())


In [ ]:
df_raw.info()


In [ ]:
df_raw.describe(include="all").T


### Expected Week 2 quality questions

- Are required columns present?
- Are numeric variables actually numeric?
- Are there missing values?
- Are there duplicate records?
- Are area, production and yield non-negative?
- Are crop names standardized?
- Are units consistent with the source publication?


## 4. Data Quality Checks

In [ ]:
quality = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "unique_count": df_raw.nunique(),
    "dtype": df_raw.dtypes.astype(str)
})
quality


In [ ]:
duplicate_count = int(df_raw.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)


In [ ]:
numeric_cols = ["area_mha", "production_mt", "yield_kg_ha"]

range_checks = pd.DataFrame({
    "min": df_raw[numeric_cols].min(),
    "max": df_raw[numeric_cols].max(),
    "negative_count": [(df_raw[c] < 0).sum() for c in numeric_cols]
})
range_checks


## 5. Cleaning and Standardization

Cleaning rules used in this notebook:

- Copy the raw dataset rather than modifying it.
- Normalize column names.
- Convert numeric fields explicitly.
- Strip whitespace from categorical fields.
- Use a reference mapping for crop names.
- Check duplicates before and after cleaning.
- Do not automatically impute production or yield.
- Flag impossible negative values.
- Preserve the original period labels because the periods are overlapping five-year normal estimates.


In [ ]:
df = df_raw.copy()

# Standardize column names.
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# Standardize text fields.
for col in ["crop", "period"]:
    df[col] = df[col].astype(str).str.strip()

# Convert numeric fields.
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()


In [ ]:
# Load and apply the controlled crop-name mapping.
crop_map = pd.read_csv(BASE / "data" / "reference" / "crop_name_mapping.csv")

df = df.merge(
    crop_map[["raw_crop", "standard_crop", "crop_group"]],
    left_on="crop",
    right_on="raw_crop",
    how="left",
    validate="many_to_one"
)

df["crop"] = df["standard_crop"]
df = df.drop(columns=["raw_crop", "standard_crop"])

print("Unmapped crops:", df["crop_group"].isna().sum())
df[["crop", "crop_group"]].drop_duplicates().sort_values(["crop_group", "crop"])


## 6. Missing Values: Decision-Based Handling

In [ ]:
missing_report = pd.DataFrame({
    "field": df.columns,
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [df[c].isna().mean() * 100 for c in df.columns]
})
missing_report


**Decision:** production and yield are not automatically filled with means. In agricultural production data, a missing observation can represent non-reporting, structural absence, or an unavailable estimate. The correct action is to investigate the source before imputation. If later modeling requires imputation, the method and assumptions must be documented separately.


## 7. Duplicate and Domain Validation

In [ ]:
# Exact duplicates after standardization.
print("Duplicates after standardization:", int(df.duplicated().sum()))

# Domain checks.
invalid = df[
    (df["area_mha"] < 0) |
    (df["production_mt"] < 0) |
    (df["yield_kg_ha"] < 0)
]

print("Rows with negative agricultural values:", len(invalid))
invalid


### Production–area–yield consistency

For compatible units, the relationship is approximately:

**Production = Area × Yield**

Because this dataset stores area in million hectares, production in million tonnes and yield in kg/ha, a direct calculation requires unit conversion. The check below is therefore used as a diagnostic, not as a blind correction rule.


In [ ]:
# Expected production in million tonnes:
# area_mha * 1e6 hectares * yield_kg_ha / 1e6 kg per tonne / 1e6 tonnes
# = area_mha * yield_kg_ha / 1000 million tonnes
df["production_from_area_yield_mt"] = (
    df["area_mha"] * df["yield_kg_ha"] / 1000
)

df["production_check_pct_diff"] = (
    (df["production_mt"] - df["production_from_area_yield_mt"])
    / df["production_mt"].replace(0, np.nan)
    * 100
)

df[[
    "crop", "period", "area_mha", "production_mt",
    "yield_kg_ha", "production_from_area_yield_mt",
    "production_check_pct_diff"
]].head(10)


The diagnostic above helps identify potential unit or definition issues. It should **not** be used to overwrite official production values, because official estimates may use source-specific estimation and rounding procedures.


## 8. Outlier and Range Screening

In [ ]:
# Simple statistical screening for unusually high values.
# This is a flagging mechanism, not automatic deletion.

def iqr_flags(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

for col in numeric_cols:
    df[f"{col}_outlier_flag"] = iqr_flags(df[col])

pd.DataFrame({
    "field": numeric_cols,
    "flagged_rows": [df[f"{c}_outlier_flag"].sum() for c in numeric_cols]
})


An outlier is not automatically an error. Large agricultural values can be legitimate for major crops or aggregate groups. Every flagged value should be checked against the original source and domain context before removal.


## 9. Derived Features

In [ ]:
# Earliest-to-latest changes within the available normal-estimate periods.
period_order = list(df["period"].drop_duplicates())

first_period = period_order[0]
last_period = period_order[-1]

first = df[df["period"] == first_period].set_index("crop")
last = df[df["period"] == last_period].set_index("crop")

change = pd.DataFrame(index=first.index)
for col in ["area_mha", "production_mt", "yield_kg_ha"]:
    change[f"{col}_pct_change"] = (
        (last[col] - first[col]) / first[col] * 100
    )

change.sort_values("production_mt_pct_change", ascending=False).round(2)


These percentage changes are **average-period comparisons**, not annual growth rates, because the normal-estimate periods overlap.


## 10. Processed Dataset

In [ ]:
# Remove diagnostic-only columns before saving the clean analytical file.
diagnostic_cols = [
    "production_from_area_yield_mt",
    "production_check_pct_diff",
    "area_mha_outlier_flag",
    "production_mt_outlier_flag",
    "yield_kg_ha_outlier_flag"
]

df_clean = df.drop(columns=diagnostic_cols)

# Final duplicate check.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_PATH, index=False)

print("Saved:", PROCESSED_PATH)
print("Rows:", len(df_clean))
df_clean.head()


## 11. Final Quality Report

In [ ]:
final_quality = pd.DataFrame({
    "check": [
        "row_count",
        "column_count",
        "missing_cells",
        "duplicate_rows",
        "negative_area",
        "negative_production",
        "negative_yield",
        "unique_crops",
        "unique_periods"
    ],
    "value": [
        len(df_clean),
        df_clean.shape[1],
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        int((df_clean["area_mha"] < 0).sum()),
        int((df_clean["production_mt"] < 0).sum()),
        int((df_clean["yield_kg_ha"] < 0).sum()),
        int(df_clean["crop"].nunique()),
        int(df_clean["period"].nunique())
    ]
})

final_quality


In [ ]:
QUALITY_PATH = BASE / "outputs" / "quality_reports" / "des_quality_report.csv"
QUALITY_PATH.parent.mkdir(parents=True, exist_ok=True)
final_quality.to_csv(QUALITY_PATH, index=False)
print("Saved:", QUALITY_PATH)


## 12. Collection and Cleaning Checklist

**Collection**
- [x] Preserve raw source file
- [x] Record source metadata
- [x] Define required fields
- [ ] Add state/district APY download
- [ ] Add climate/rainfall dataset
- [ ] Add Agmarknet market dataset

**Cleaning**
- [x] Schema inspection
- [x] Data-type conversion
- [x] Missing-value profiling
- [x] Duplicate checks
- [x] Unit/domain checks
- [x] Crop-name mapping
- [x] Outlier flagging
- [x] Production-area-yield diagnostic
- [x] Clean dataset export
- [x] Quality-report export

**Next implementation step:** repeat this workflow for district/state APY data, then add validated climate and market tables using documented join keys.


## 13. Official Sources

- DES APY Reports: https://data.desagri.gov.in/website/crops-apy-report-web
- DES APY Query Report: https://data.desagri.gov.in/website/apy-query-report-web
- India OGD district-wise crop production: https://www.data.gov.in/catalog/district-wise-season-wise-crop-production-statistics-0
- Agmarknet/e-NAM price dashboard: https://enam.gov.in/web/dashboard/agmarknet

DES currently provides APY reporting by state, district, crop, season and year. citeturn0search0turn0search1 The OGD catalog provides district-wise, crop-wise, season-wise and year-wise covered area and production data. citeturn0search17 The Agmarknet dashboard exposes market, commodity, arrivals, price and date fields for agricultural market analysis. citeturn0search14
